<a href="https://colab.research.google.com/github/pablobelmiro/olist_exploration/blob/main/02_visualizacoes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Olist E-Commerce: Capítulo 2, Visualizações Exploratórias

Continuação do Capítulo 1 (modelo relacional + EDA). Esse notebook parte do zero de novo (carrega os 9 CSVs e remonta o dataframe mestre), pra ficar autocontido, mas assume que você já leu o capítulo anterior.

Sobe os mesmos 9 CSVs pro mesmo diretório desse notebook. Não precisa de GPU, é só pandas.

In [1]:
import pandas as pd
import json

orders = pd.read_csv('olist_orders_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
category_translation = pd.read_csv('product_category_name_translation.csv')

for col in ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']:
    orders[col] = pd.to_datetime(orders[col])

produtos_com_categoria_en = products.merge(category_translation, on='product_category_name', how='left')

mestre = (
    order_items
    .merge(orders, on='order_id', how='left')
    .merge(customers, on='customer_id', how='left')
    .merge(produtos_com_categoria_en, on='product_id', how='left')
    .merge(sellers, on='seller_id', how='left')
    .merge(payments, on='order_id', how='left')
    .merge(reviews, on='order_id', how='left')
)

print(f"dataframe mestre: {mestre.shape}")

dataframe mestre: (118310, 40)


## 1. Temporal: dia da semana x hora da compra

`weekday` vai de 0 (segunda) a 6 (domingo).

In [2]:
orders_com_hora = orders.dropna(subset=['order_purchase_timestamp']).copy()
orders_com_hora['weekday'] = orders_com_hora['order_purchase_timestamp'].dt.weekday
orders_com_hora['hour'] = orders_com_hora['order_purchase_timestamp'].dt.hour

weekday_hour = (
    orders_com_hora
    .groupby(['weekday', 'hour'])['order_id']
    .nunique()
    .reset_index()
    .rename(columns={'order_id': 'orders'})
)

print(f"células: {len(weekday_hour)} (deveria ser 7 x 24 = 168)")
print(f"soma total: {weekday_hour['orders'].sum()} (deveria bater com o total de pedidos com data: {orders_com_hora['order_id'].nunique()})")

weekday_hour_json = [
    {'weekday': int(r.weekday), 'hour': int(r.hour), 'orders': int(r.orders)}
    for r in weekday_hour.itertuples()
]

with open('weekday-hour.json', 'w', encoding='utf-8') as f:
    json.dump(weekday_hour_json, f, ensure_ascii=False, indent=2)

print(weekday_hour_json[:5])

células: 168 (deveria ser 7 x 24 = 168)
soma total: 99441 (deveria bater com o total de pedidos com data: 99441)
[{'weekday': 0, 'hour': 0, 'orders': 328}, {'weekday': 0, 'hour': 1, 'orders': 134}, {'weekday': 0, 'hour': 2, 'orders': 66}, {'weekday': 0, 'hour': 3, 'orders': 36}, {'weekday': 0, 'hour': 4, 'orders': 21}]


## 2. Geográfico: pedidos, frete e atraso por estado

Agrupa por `customer_state` (estado do cliente, não do vendedor). `avg_delivery_days` é o tempo real de entrega (compra até entrega), não o atraso contra a estimativa, isso fica pro capítulo 3.

In [3]:
orders_com_cliente = orders.merge(customers, on='customer_id', how='left')

frete_por_pedido = order_items.groupby('order_id')['freight_value'].sum().reset_index()
orders_com_cliente = orders_com_cliente.merge(frete_por_pedido, on='order_id', how='left')

orders_com_cliente['delivery_days'] = (
    orders_com_cliente['order_delivered_customer_date'] - orders_com_cliente['order_purchase_timestamp']
).dt.days

por_estado = orders_com_cliente.groupby('customer_state').agg(
    orders=('order_id', 'nunique'),
    avg_freight=('freight_value', 'mean'),
    avg_delivery_days=('delivery_days', 'mean'),
).reset_index()

print(f"estados: {len(por_estado)}")
print(por_estado.sort_values('avg_delivery_days', ascending=False).to_string(index=False))

por_estado_json = [
    {
        'state': r.customer_state,
        'orders': int(r.orders),
        'avg_freight': round(float(r.avg_freight), 2),
        'avg_delivery_days': round(float(r.avg_delivery_days), 1),
    }
    for r in por_estado.itertuples()
]

with open('orders-by-state.json', 'w', encoding='utf-8') as f:
    json.dump(por_estado_json, f, ensure_ascii=False, indent=2)

estados: 27
customer_state  orders  avg_freight  avg_delivery_days
            RR      46    48.591087          28.975610
            AP      68    41.007353          26.731343
            AM     148    37.271361          25.986207
            AL     413    38.721630          24.040302
            PA     975    39.896186          23.316068
            MA     747    42.599689          21.117155
            SE     350    40.902812          21.029851
            CE    1336    36.436767          20.817826
            AC      81    45.515432          20.637500
            PB     536    48.345357          19.953578
            PI     495    43.038945          18.993697
            RO     253    46.224211          18.913580
            BA    3380    29.826289          18.866400
            RN     485    39.128838          18.824895
            PE    1652    36.073823          17.965474
            MT     907    32.907453          17.593679
            TO     280    42.052616          17.22627

## 3. Categoria: top 15 por receita

Receita = soma de `price` (não inclui frete) por categoria, usando o nome em inglês já traduzido.

In [4]:
top_categorias = (
    mestre
    .dropna(subset=['product_category_name_english'])
    .groupby('product_category_name_english')['price']
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)

print(top_categorias.to_string(index=False))

top_categorias_json = [
    {'category': r.product_category_name_english, 'revenue': round(float(r.price), 2)}
    for r in top_categorias.itertuples()
]

with open('top-categories.json', 'w', encoding='utf-8') as f:
    json.dump(top_categorias_json, f, ensure_ascii=False, indent=2)

product_category_name_english      price
                health_beauty 1301947.97
                watches_gifts 1254322.95
               bed_bath_table 1107249.09
               sports_leisure 1029603.88
        computers_accessories  950053.69
              furniture_decor  772096.17
                   housewares  668880.94
                   cool_stuff  664637.13
                         auto  618395.50
                 garden_tools  519473.33
                         toys  501118.39
                         baby  434832.19
                    perfumery  415055.76
                    telephony  339571.03
             office_furniture  287422.75


## 4. Pagamento: distribuição por tipo

Descarta `not_defined` (só umas poucas linhas, ruído, não vale o pedaço de pizza).

In [5]:
tipos_pagamento = (
    payments[payments['payment_type'] != 'not_defined']
    .groupby('payment_type')['order_id']
    .nunique()
    .reset_index()
    .rename(columns={'order_id': 'count'})
    .sort_values('count', ascending=False)
)

print(tipos_pagamento.to_string(index=False))

tipos_pagamento_json = [
    {'type': r.payment_type, 'count': int(r.count)}
    for r in tipos_pagamento.itertuples()
]

with open('payment-types.json', 'w', encoding='utf-8') as f:
    json.dump(tipos_pagamento_json, f, ensure_ascii=False, indent=2)

payment_type  count
 credit_card  76505
      boleto  19784
     voucher   3866
  debit_card   1528


## 5. Satisfação: distribuição de nota, e nota x atraso

Duas peças aqui: a distribuição pura de `review_score` (1 a 5), e uma amostra real de atraso (dias, negativo é entrega adiantada) contra a nota dada, pra ver se tem relação visual antes de qualquer modelo.

In [6]:
distribuicao_notas = (
    reviews
    .groupby('review_score')['review_id']
    .count()
    .reset_index()
    .rename(columns={'review_id': 'count'})
)

print(distribuicao_notas.to_string(index=False))

distribuicao_notas_json = [
    {'score': int(r.review_score), 'count': int(r.count)}
    for r in distribuicao_notas.itertuples()
]

with open('review-score-distribution.json', 'w', encoding='utf-8') as f:
    json.dump(distribuicao_notas_json, f, ensure_ascii=False, indent=2)

 review_score  count
            1  11424
            2   3151
            3   8179
            4  19142
            5  57328


In [7]:
nota_atraso = (
    orders[['order_id', 'order_delivered_customer_date', 'order_estimated_delivery_date']]
    .merge(reviews[['order_id', 'review_score']], on='order_id', how='inner')
    .dropna(subset=['order_delivered_customer_date', 'order_estimated_delivery_date', 'review_score'])
)

nota_atraso['delay_days'] = (
    nota_atraso['order_delivered_customer_date'] - nota_atraso['order_estimated_delivery_date']
).dt.days

print(f"pedidos com entrega + nota: {len(nota_atraso)}")

# amostra de 400 pontos, seed fixa pra reprodutibilidade
amostra = nota_atraso.sample(n=400, random_state=42)

amostra_json = [
    {'delay_days': int(r.delay_days), 'review_score': int(r.review_score)}
    for r in amostra.itertuples()
]

with open('review-vs-delay-sample.json', 'w', encoding='utf-8') as f:
    json.dump(amostra_json, f, ensure_ascii=False, indent=2)

print(amostra_json[:5])

pedidos com entrega + nota: 96359
[{'delay_days': -21, 'review_score': 5}, {'delay_days': -7, 'review_score': 1}, {'delay_days': -15, 'review_score': 5}, {'delay_days': -18, 'review_score': 3}, {'delay_days': -17, 'review_score': 1}]


## 6. Correlação geral entre as features numéricas

`price`, `freight_value`, `product_weight_g`, `payment_value`, `review_score`, e `delivery_days` calculado na hora. Correlação de Pearson padrão do pandas.

In [8]:
features_numericas = mestre[['price', 'freight_value', 'product_weight_g', 'payment_value', 'review_score']].copy()
features_numericas['delivery_days'] = (
    mestre['order_delivered_customer_date'] - mestre['order_purchase_timestamp']
).dt.days
features_numericas = features_numericas.dropna()

print(f"linhas usadas na correlação: {len(features_numericas)}")

matriz_corr = features_numericas.corr()
print(matriz_corr.round(2))

features_lista = list(matriz_corr.columns)
celulas = [
    {'x': fx, 'y': fy, 'value': round(float(matriz_corr.loc[fy, fx]), 2)}
    for fx in features_lista
    for fy in features_lista
]

correlacao_json = {'features': features_lista, 'cells': celulas}

with open('correlation-matrix.json', 'w', encoding='utf-8') as f:
    json.dump(correlacao_json, f, ensure_ascii=False, indent=2)

linhas usadas na correlação: 114838
                  price  freight_value  product_weight_g  payment_value  \
price              1.00           0.41              0.34           0.74   
freight_value      0.41           1.00              0.61           0.37   
product_weight_g   0.34           0.61              1.00           0.31   
payment_value      0.74           0.37              0.31           1.00   
review_score       0.00          -0.03             -0.03          -0.08   
delivery_days      0.06           0.21              0.08           0.06   

                  review_score  delivery_days  
price                     0.00           0.06  
freight_value            -0.03           0.21  
product_weight_g         -0.03           0.08  
payment_value            -0.08           0.06  
review_score              1.00          -0.30  
delivery_days            -0.30           1.00  
